# Evaluation and ablations

Load a generated prediction file and compute metrics and a confusion matrix.

In [1]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate import load_predictions, compute_metrics, build_confusion_matrix, build_classification_report
import matplotlib.pyplot as plt

prediction_path = PROJECT_ROOT / 'results' / 'predictions' / 'model.csv'
if not prediction_path.exists():
    raise FileNotFoundError(f'Prediction file not found: {prediction_path}. Run the classical baseline or provide Person A predictions first.')
predictions = load_predictions(prediction_path)
metrics = compute_metrics(predictions)
confusion = build_confusion_matrix(predictions)
figures_dir = PROJECT_ROOT / 'figures'
results_dir = PROJECT_ROOT / 'results'
figures_dir.mkdir(parents=True, exist_ok=True)
report = build_classification_report(predictions)
pd.DataFrame([metrics]).to_csv(results_dir / 'metrics.csv', index=False)
report.to_csv(results_dir / 'classification_report.csv')
pd.DataFrame(confusion, index=['negative', 'neutral', 'positive'], columns=['negative', 'neutral', 'positive']).to_csv(figures_dir / 'confusion_matrix.csv')
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(confusion, cmap='Blues')
ax.set_xticks(range(3), ['negative', 'neutral', 'positive'])
ax.set_yticks(range(3), ['negative', 'neutral', 'positive'])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion matrix')
for row in range(3):
    for col in range(3):
        ax.text(col, row, int(confusion[row, col]), ha='center', va='center')
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(figures_dir / 'confusion_matrix.png', dpi=150)
plt.close(fig)
metrics, confusion


({'accuracy': 0.8716666666666667,
  'macro_f1': 0.8719155477320047,
  'weighted_f1': 0.8716173700248004},
 array([[794,  55,  41],
        [ 78, 960,  90],
        [ 52,  69, 861]]))